<div style="background-color: #ffffff; color: #000000; padding: 30px;">
<img src="../media/images/kisz_logo.png" width="192" height="69" align="right" style="margin-right: 50px; margin-bottom: 50px;">
<h1>Time Series Analysis and Forecasting</h1>
</div>

<div style="background-color: #f6a800; color: #ffffff; padding: 10px;">
<h2>Part B: Statistical Forecasting</h2>
<h2>Notebook B04: Probabilistic Forecasting</h2>
</div>

Every forecast so far has been a single number per month, which quietly asserts that we know exactly what
will happen. We do not, and saying so is usually the more useful answer.

"Next January will average 1.5 °C" is a claim that will be wrong. "Next January will fall between -1 and
4 °C, with 80% probability" is a claim that can be *right*, and it is the one someone planning gas storage
can actually use.

This notebook produces those ranges, checks whether they are honest, and scores them. Checking is the part
people skip: an interval that is wrong about how often it is wrong is worse than no interval at all,
because it invites confident decisions on false precision.

---

**Contents**

1. [Imports and Data Loading](#1.-Imports-and-Data-Loading)
2. [From a Point to a Distribution](#2.-From-a-Point-to-a-Distribution)
3. [Prediction Intervals from the Model](#3.-Prediction-Intervals-from-the-Model)
4. [Coverage and Sharpness](#4.-Coverage-and-Sharpness)
5. [Conformal Prediction](#5.-Conformal-Prediction)
6. [Scoring Rules](#6.-Scoring-Rules)
7. [Visualising Uncertainty](#7.-Visualising-Uncertainty)
8. [What to Take Away](#8.-What-to-Take-Away)

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="1.-Imports-and-Data-Loading">1. Imports and Data Loading</h3>
</div>

In [ ]:
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from statsmodels.tsa.statespace.sarimax import SARIMAX

import nb_config

sns.set_theme(style="whitegrid")

The same temperature series as the rest of Part B. We use the SARIMA specification from Notebook
[B02](./B02_ARIMA_models.ipynb), in its cheaper `(1,0,0)(0,1,1,12)` form, because this notebook fits it
two dozen times and the difference in accuracy is small.

In [ ]:
series = pd.read_parquet(nb_config.CDC_TEMP_PATH)["Brandenburg/Berlin"].asfreq("MS")

ORDER = (1, 0, 0)
SEASONAL_ORDER = (0, 1, 1, 12)
HORIZON = 12

train = series.iloc[:-HORIZON]
test = series.iloc[-HORIZON:]


def fit_sarima(training_data):
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        return SARIMAX(training_data, order=ORDER, seasonal_order=SEASONAL_ORDER).fit(disp=False)


print(f"Train: {len(train)} months, Test: {len(test)} months")

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="2.-From-a-Point-to-a-Distribution">2. From a Point to a Distribution</h3>
</div>

A probabilistic forecast replaces the single number with a **distribution** over what might happen. Three
words for the same idea at different resolutions:

- A **quantile** is one point of that distribution. The 5th percentile is the value the outcome should
  fall below 5% of the time.
- A **prediction interval** is a pair of quantiles. An 80% interval runs from the 10th to the 90th
  percentile, so 10% of outcomes should fall below it and 10% above.
- The **predictive distribution** is the whole thing, from which any quantile or interval can be read.

Note the word *should*. A stated 80% interval is a promise about long-run frequency, and section 4 is
about holding it to that promise.

There is more than one way to produce these, and the choice usually follows from the model you already
have:

| Approach | How | Where it appears |
|---|---|---|
| **Model-based** | The model's own error distribution | This notebook, section 3 |
| **Conformal** | Empirical quantiles of past errors | This notebook, section 5 |
| **Quantile regression** | Fit each quantile directly | Part C |
| **Ensembles** | Spread across many models or simulations | Parts C and D |
| **Bayesian** | Distributions over the parameters too | Beyond this course |

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="3.-Prediction-Intervals-from-the-Model">3. Prediction Intervals from the Model</h3>
</div>

A statistical model like SARIMA carries a distribution with it. It assumes the errors are normal with a
variance it estimates, and it can propagate that forward to say how uncertain each future point is.
`statsmodels` exposes this through `get_forecast`, which returns the mean, the standard error, and
intervals at any level you ask for.

In [ ]:
model = fit_sarima(train)
forecast = model.get_forecast(HORIZON)

intervals = pd.DataFrame({
    "actual": test,
    "forecast": forecast.predicted_mean,
    "std error": forecast.se_mean,
})

for level in (0.80, 0.95):
    bounds = forecast.conf_int(alpha=1 - level)
    intervals[f"lo{level:.0%}"] = bounds.iloc[:, 0].values
    intervals[f"hi{level:.0%}"] = bounds.iloc[:, 1].values

intervals.round(2)

Two things to notice.

The **standard error barely grows** with the horizon, from 1.91 to about 1.97 over a year. That is
unusual, and it is a property of this series rather than of SARIMA: with a seasonal difference doing the
work, next August is pinned down by the seasonal structure almost as well as next month is. On a series
with a trend or a random walk the intervals would widen steadily, as the fan chart in section 7 expects.

The **80% interval is much narrower than the 95%**, roughly 5 °C against 7.7 °C. Demanding more certainty
costs width, and that trade-off is the entire subject of section 4.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4.5))

ax.plot(train["2021":], color="steelblue", linewidth=1.2, label="Train")
ax.plot(test, color="black", linewidth=1.8, marker="o", markersize=5, label="Actual")
ax.plot(intervals["forecast"], color="crimson", linewidth=1.5, linestyle="--", label="Forecast")

ax.fill_between(intervals.index, intervals["lo95%"], intervals["hi95%"],
                color="crimson", alpha=0.12, label="95% interval")
ax.fill_between(intervals.index, intervals["lo80%"], intervals["hi80%"],
                color="crimson", alpha=0.22, label="80% interval")

ax.axvline(test.index[0], color="gray", linestyle="--", linewidth=1.0)
ax.set_title("SARIMA forecast with prediction intervals", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Temperature (°C)")
ax.legend(loc="upper left")
ax.grid(axis="y", linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="4.-Coverage-and-Sharpness">4. Coverage and Sharpness</h3>
</div>

An interval makes a testable promise: a 95% interval should contain the truth about 95% of the time. Two
properties matter, and they pull against each other.

**Calibration** asks whether the promise is kept. Compare the **empirical coverage**, the fraction of
outcomes that actually landed inside, against the nominal level. Coverage below the nominal level means
the model is **overconfident**, which is the dangerous direction.

**Sharpness** asks how narrow the intervals are. Narrow is better, but only while calibration holds: the
interval from minus infinity to plus infinity has perfect coverage and no value at all.

A single 12-month test window gives twelve observations, which is nowhere near enough to measure a
frequency. So we borrow the rolling-origin machinery from Notebook
[A06](./A06_Evaluating_models.ipynb) and collect forecasts from many origins.

> **This cell fits the model 24 times and takes about half a minute.**

In [ ]:
def collect_forecasts(series, n_origins=24, horizon=HORIZON, step=12):
    """Forecast from many origins, keeping the mean and standard error of each point."""
    records = []

    for k in range(n_origins):
        end = len(series) - horizon - (n_origins - 1 - k) * step
        training_data, actual = series.iloc[:end], series.iloc[end:end + horizon]

        forecast = fit_sarima(training_data).get_forecast(horizon)

        for h in range(horizon):
            records.append({
                "origin": k,
                "horizon": h + 1,
                "actual": actual.iloc[h],
                "mean": forecast.predicted_mean.iloc[h],
                "std": forecast.se_mean.iloc[h],
            })

    return pd.DataFrame(records)


evaluations = collect_forecasts(series)
evaluations["error"] = evaluations["actual"] - evaluations["mean"]

print(f"{evaluations['origin'].nunique()} origins x {HORIZON} horizons "
      f"= {len(evaluations)} forecast points")

In [ ]:
def coverage_report(frame, levels=(0.50, 0.80, 0.95)):
    """Empirical coverage and mean interval width at each nominal level."""
    rows = []
    for level in levels:
        z = stats.norm.ppf(0.5 + level / 2)
        lower = frame["mean"] - z * frame["std"]
        upper = frame["mean"] + z * frame["std"]
        inside = (frame["actual"] >= lower) & (frame["actual"] <= upper)
        rows.append({
            "Nominal": f"{level:.0%}",
            "Empirical coverage": inside.mean(),
            "Mean width (°C)": (upper - lower).mean(),
        })
    return pd.DataFrame(rows)


coverage_report(evaluations).round(3)

These intervals are honest. 52% against a promised 50%, 78% against 80%, 94% against 95%: all within
what you would expect from 288 observations. The model's distributional assumption is doing real work
here, and we can trust its intervals on this series.

Well-calibrated is not the default, though, and it is worth seeing what the failure looks like. The cell
below keeps exactly the same point forecasts and deliberately understates the uncertainty, as a model
that had underestimated its own error variance would.

In [ ]:
overconfident = evaluations.copy()
overconfident["std"] = overconfident["std"] * 0.5   # claim half the uncertainty

comparison = pd.concat([
    coverage_report(evaluations).assign(Intervals="Model"),
    coverage_report(overconfident).assign(Intervals="Overconfident (half width)"),
])

comparison.pivot(index="Nominal", columns="Intervals",
                 values=["Empirical coverage", "Mean width (°C)"]).round(3)

The overconfident intervals are half as wide, which by the sharpness criterion alone looks like an
improvement. Their 95% interval contains the truth 68% of the time.

That is the failure mode worth remembering. **Sharpness without calibration is not a better forecast, it
is a worse one presented more confidently**, and a summary table of interval widths would have rewarded
it. Always report coverage next to width.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

levels = np.linspace(0.05, 0.99, 30)
for frame, label, colour in [(evaluations, "Model", "steelblue"),
                             (overconfident, "Overconfident", "crimson")]:
    empirical = []
    for level in levels:
        z = stats.norm.ppf(0.5 + level / 2)
        inside = (frame["actual"] - frame["mean"]).abs() <= z * frame["std"]
        empirical.append(inside.mean())
    axes[0].plot(levels, empirical, color=colour, linewidth=1.8, label=label)

axes[0].plot([0, 1], [0, 1], color="black", linestyle="--", linewidth=1.0, label="Perfect calibration")
axes[0].set_title("Calibration plot", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Nominal coverage")
axes[0].set_ylabel("Empirical coverage")
axes[0].legend(loc="upper left")

by_horizon = evaluations.groupby("horizon").agg(
    width=("std", lambda s: 2 * stats.norm.ppf(0.9) * s.mean()),
    absolute_error=("error", lambda e: e.abs().mean()),
)
axes[1].plot(by_horizon.index, by_horizon["width"], color="steelblue",
             marker="o", markersize=4, label="80% interval width")
axes[1].plot(by_horizon.index, by_horizon["absolute_error"], color="darkorange",
             marker="s", markersize=4, label="Mean absolute error")
axes[1].set_title("Uncertainty by horizon", fontsize=13, fontweight="bold")
axes[1].set_xlabel("Months ahead")
axes[1].set_ylabel("°C")
axes[1].legend(loc="upper left")

for ax in axes:
    ax.grid(linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

The calibration plot is the single most useful diagnostic in this notebook. A well-calibrated forecast
traces the diagonal; the overconfident one sags below it everywhere, and the gap is the size of the lie.

The right-hand panel shows why the intervals here stay flat across the horizon: the error does not grow
either. Forecasting next December is no harder than forecasting next month, because the seasonal
structure does the work in both cases.

**Exercise.** Build the same calibration plot for intervals that are too *wide* (multiply `std` by 1.5). Where does the curve sit relative to the diagonal, and why is this failure less dangerous than overconfidence, even though it is equally miscalibrated?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="5.-Conformal-Prediction">5. Conformal Prediction</h3>
</div>

The intervals above are only as good as the assumption behind them: that the errors are normal, with the
variance the model estimated. On this series that assumption holds. Often it does not, and for most
machine learning models there is no error distribution to appeal to in the first place.

**Conformal prediction** sidesteps the question. Instead of assuming a distribution, measure the errors
the model actually made on data it did not train on, and use their empirical quantiles as the interval
width. If you want 80% coverage, take the 80th percentile of past absolute errors.

The guarantee is **distribution-free**: given exchangeable data, coverage holds whatever the error
distribution looks like. Time series are not exchangeable, so the guarantee is approximate here, but the
method remains a practical and honest way to size an interval.

Errors grow with the horizon in general, so we compute a separate quantile for each horizon. We split
the rolling-origin results into a calibration half and a test half, and never let the test half
influence the width.

In [ ]:
CALIBRATION_ORIGINS = 16

calibration = evaluations[evaluations["origin"] < CALIBRATION_ORIGINS]
held_out = evaluations[evaluations["origin"] >= CALIBRATION_ORIGINS]


def conformal_widths(calibration, level):
    """Half-width per horizon: the empirical quantile of absolute calibration errors."""
    return (
        calibration.assign(absolute_error=calibration["error"].abs())
        .groupby("horizon")["absolute_error"]
        .quantile(level, interpolation="higher")
    )


rows = []
for level in (0.80, 0.95):
    z = stats.norm.ppf(0.5 + level / 2)
    model_inside = (held_out["error"].abs() <= z * held_out["std"])

    half_width = held_out["horizon"].map(conformal_widths(calibration, level))
    conformal_inside = (held_out["error"].abs() <= half_width)

    rows.append({
        "Nominal": f"{level:.0%}",
        "Model coverage": model_inside.mean(),
        "Model width": (2 * z * held_out["std"]).mean(),
        "Conformal coverage": conformal_inside.mean(),
        "Conformal width": (2 * half_width).mean(),
    })

print(f"Calibrated on {CALIBRATION_ORIGINS} origins, evaluated on "
      f"{held_out['origin'].nunique()} held-out origins")
pd.DataFrame(rows).round(3)

The two approaches agree, which is the result we should have expected and is worth stating plainly: on a
series whose errors really are close to normal, a method that assumes normality and a method that assumes
nothing arrive at the same place. The conformal intervals are marginally narrower at 80% and marginally
wider at 95%, and neither difference means anything.

That agreement is itself useful. It is evidence that the model's distributional assumption is sound here,
arrived at without trusting the assumption. When the two *disagree*, the model-based interval is the one
to distrust.

Conformal prediction is worth knowing for two situations this series does not show. When the errors are
skewed or heavy-tailed, a normal interval will be wrong in a direction you can predict but not easily
fix. And when the model has no error distribution at all, which covers most of Parts C and D, conformal
is one of the few principled options available.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 4.5))

sorted_errors = np.sort(calibration["error"].abs())
empirical_cdf = np.arange(1, len(sorted_errors) + 1) / len(sorted_errors)
ax.plot(sorted_errors, empirical_cdf, color="steelblue", linewidth=1.8,
        label="Observed absolute errors")

sigma = calibration["std"].mean()
grid = np.linspace(0, sorted_errors.max(), 200)
ax.plot(grid, 2 * stats.norm.cdf(grid / sigma) - 1, color="crimson", linewidth=1.8,
        linestyle="--", label="Normal assumption")

for level, colour in [(0.80, "darkorange"), (0.95, "seagreen")]:
    ax.axhline(level, color=colour, linewidth=0.9, linestyle=":")
    ax.text(sorted_errors.max() * 0.82, level + 0.015, f"{level:.0%}", color=colour, fontsize=10)

ax.set_title("Where the interval width comes from", fontsize=14, fontweight="bold")
ax.set_xlabel("Absolute error (°C)")
ax.set_ylabel("Fraction of errors at or below")
ax.legend(loc="lower right")
ax.grid(linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

Read a width off the horizontal lines: where each curve crosses 80% is the half-width that method
assigns. The two curves track each other closely, which is the same conclusion as the table, drawn
instead of tabulated. A visible gap between them would be a warning that the normal assumption is
distorting the intervals.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="6.-Scoring-Rules">6. Scoring Rules</h3>
</div>

Coverage checks one level at a time and says nothing about whether the forecast put its probability mass
in the right place. A **proper scoring rule** grades the whole distribution with one number, and is
constructed so that it is optimised by reporting your true beliefs: you cannot score better by
exaggerating or hedging.

**Pinball loss** (quantile loss) grades a single quantile $\tau$:

$$L_\tau(y, q) = \begin{cases} \tau(y - q) & y \ge q \\ (1-\tau)(q - y) & y < q \end{cases}$$

The asymmetry is the point. For the 90th percentile, being below the outcome is penalised nine times as
heavily as being above it, which is exactly the incentive that makes an honest 90th percentile optimal.

**CRPS**, the continuous ranked probability score, integrates the pinball loss over every quantile at
once, giving one number for the whole predictive distribution. It is in the units of the data, and for a
point forecast it reduces to the absolute error, so it can be compared directly with the MAE figures from
earlier notebooks.

In [ ]:
def pinball_loss(actual, quantile_forecast, tau):
    """Loss for a single quantile. Under-forecasting costs tau, over-forecasting 1 - tau."""
    difference = np.asarray(actual) - np.asarray(quantile_forecast)
    return float(np.mean(np.maximum(tau * difference, (tau - 1) * difference)))


def crps_normal(actual, mean, std):
    """CRPS for a normal predictive distribution, in closed form."""
    actual, mean, std = map(np.asarray, (actual, mean, std))
    z = (actual - mean) / std
    return float(np.mean(
        std * (z * (2 * stats.norm.cdf(z) - 1) + 2 * stats.norm.pdf(z) - 1 / np.sqrt(np.pi))
    ))

A closed-form expression is easy to get subtly wrong, so it is worth checking against a definition that
is obviously correct. CRPS can also be written as an expectation over samples from the predictive
distribution, $E|X - y| - \tfrac{1}{2}E|X - X'|$, which we can estimate by simulation.

In [ ]:
rng = np.random.default_rng(0)
example = {"actual": 3.0, "mean": 2.0, "std": 1.5}

draws = rng.normal(example["mean"], example["std"], 200_000)
independent_draws = rng.normal(example["mean"], example["std"], 200_000)
simulated = (
    np.mean(np.abs(draws - example["actual"]))
    - 0.5 * np.mean(np.abs(draws - independent_draws))
)

print(f"Closed form: {crps_normal([example['actual']], [example['mean']], [example['std']]):.4f}")
print(f"Simulated:   {simulated:.4f}")

In [ ]:
scores = []

for tau in (0.1, 0.5, 0.9):
    quantile_forecast = evaluations["mean"] + stats.norm.ppf(tau) * evaluations["std"]
    scores.append({
        "Score": f"Pinball loss (tau = {tau})",
        "Model": pinball_loss(evaluations["actual"], quantile_forecast, tau),
        "Overconfident": pinball_loss(
            evaluations["actual"],
            evaluations["mean"] + stats.norm.ppf(tau) * overconfident["std"],
            tau,
        ),
    })

scores.append({
    "Score": "CRPS",
    "Model": crps_normal(evaluations["actual"], evaluations["mean"], evaluations["std"]),
    "Overconfident": crps_normal(evaluations["actual"], evaluations["mean"], overconfident["std"]),
})

scores.append({
    "Score": "MAE (point forecast only)",
    "Model": evaluations["error"].abs().mean(),
    "Overconfident": evaluations["error"].abs().mean(),
})

pd.DataFrame(scores).round(3)

The last two rows are the argument for using a proper scoring rule at all.

**MAE cannot tell the two forecasts apart.** Both have identical point forecasts, so the metric we have
relied on for six notebooks scores them equally, while one of them is lying about its uncertainty.

**CRPS can.** It is worse for the overconfident version, because it grades the whole distribution and
notices that too much probability mass was concentrated near the mean.

The pinball rows repay a closer look, because they do not all point the same way. At $\tau = 0.5$ the two
are identical, which is expected: shrinking the spread does not move the median. At $\tau = 0.9$ the
overconfident forecast is clearly worse, its too-low 90th percentile penalised nine times over whenever
the outcome lands above it. But at $\tau = 0.1$ the overconfident forecast scores **better**.

That is not a mistake in the table, it is the limitation of grading one quantile at a time. Pinball loss
at a single $\tau$ is a proper scoring rule *for that quantile only*, and a distribution that is wrong
overall can still place one of its quantiles well, here by luck and a touch of skew in the errors. Judge
a whole distribution by a single quantile and you can be led anywhere.

CRPS integrates the pinball loss across every $\tau$, which is exactly what makes it the right summary:
no single lucky quantile can rescue a distribution that is wrong. So report MAE or RMSE for a point
forecast, and CRPS when the forecast is a distribution.

**Exercise.** The overconfident forecast claims half the true uncertainty. Find the multiplier that minimises CRPS by trying a range of values between 0.5 and 2.0. Does the best score land at 1.0, and what does it mean if it does not?

In [ ]:
# Your solution here


---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="7.-Visualising-Uncertainty">7. Visualising Uncertainty</h3>
</div>

Three ways to show uncertainty, each answering a different question.

A **fan chart** shades several intervals at once, so the reader sees the whole distribution widening (or
not) over time. A **density plot** at a single horizon answers "what might next January be?" A
**calibration plot**, which we have already used, answers "should I believe any of this?"

The fan chart is the one that goes in the report, so it is worth making well: nested bands at increasing
levels, with the darkest in the middle.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 4.5),
                         gridspec_kw={"width_ratios": [2, 1]})

# Fan chart
axes[0].plot(train["2022":], color="steelblue", linewidth=1.2, label="Observed")

fan_levels = [0.50, 0.80, 0.95]
for level, alpha in zip(fan_levels, [0.45, 0.28, 0.15]):
    bounds = forecast.conf_int(alpha=1 - level)
    axes[0].fill_between(test.index, bounds.iloc[:, 0], bounds.iloc[:, 1],
                         color="crimson", alpha=alpha, label=f"{level:.0%}")

axes[0].plot(intervals["forecast"], color="crimson", linewidth=1.5, linestyle="--")
axes[0].plot(test, color="black", linewidth=1.5, marker="o", markersize=4, label="Actual")
axes[0].axvline(test.index[0], color="gray", linestyle="--", linewidth=1.0)
axes[0].set_title("Fan chart", fontsize=13, fontweight="bold")
axes[0].set_ylabel("Temperature (°C)")
axes[0].legend(loc="upper left", fontsize=9)

# Predictive density for one month
target_month = 6   # six months ahead
mean, std = intervals["forecast"].iloc[target_month - 1], intervals["std error"].iloc[target_month - 1]
actual_value = intervals["actual"].iloc[target_month - 1]

grid = np.linspace(mean - 4 * std, mean + 4 * std, 300)
axes[1].fill_between(grid, stats.norm.pdf(grid, mean, std), color="crimson", alpha=0.25)
axes[1].plot(grid, stats.norm.pdf(grid, mean, std), color="crimson", linewidth=1.8)
axes[1].axvline(actual_value, color="black", linewidth=1.8, label=f"Actual ({actual_value:.1f} °C)")
axes[1].axvline(mean, color="crimson", linewidth=1.2, linestyle="--", label=f"Forecast ({mean:.1f} °C)")
axes[1].set_title(f"Predictive density, {target_month} months ahead",
                  fontsize=13, fontweight="bold")
axes[1].set_xlabel("Temperature (°C)")
axes[1].set_ylabel("Density")
axes[1].legend(loc="upper left", fontsize=9)

for ax in axes:
    ax.grid(linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()

The fan is unusually flat, for the reason established in section 3: on a strongly seasonal series with no
trend, uncertainty a year out is barely worse than uncertainty next month. On most series the fan widens
noticeably, and a fan that stays flat when it should widen is a sign the model has not accounted for
something.

The density panel is the honest picture of a single forecast. The actual value sits comfortably inside
the bulk, which is what a well-calibrated forecast looks like for one observation. It is also all that
one observation can tell you, which is why section 4 needed 288 of them.

---
<div style="background-color: #dd6108; color: #ffffff; padding: 10px;">
<h3 id="8.-What-to-Take-Away">8. What to Take Away</h3>
</div>

**A point forecast is a distribution with the uncertainty deleted.** Decisions that depend on the risk of
an outcome, rather than its average, need the distribution.

**Calibration first, sharpness second.** An interval is a promise about frequency. Check it by measuring
coverage over many forecasts, and treat narrow intervals as an achievement only once coverage holds.
Overconfidence is the dangerous failure, because it looks like precision.

**One split will not do.** Coverage is a frequency, and frequencies need samples. This notebook needed
288 forecast points to say anything, from a test window that on its own offered twelve.

**Conformal prediction when you cannot trust the assumption.** It assumes nothing about the error
distribution, agreed with the model here, and works for models that have no error distribution at all,
which is most of what follows in Parts C and D.

**Score distributions with CRPS, not MAE.** MAE scored our honest and dishonest forecasts identically.
CRPS separated them.

A final caution. Everything in this notebook quantifies the uncertainty *the model knows about*: noise
around a structure it has assumed is correct. It cannot price in the structure being wrong, and that is
usually the larger risk. A well-calibrated interval from a misspecified model is precisely stated and
still misleading.

---

That completes Part B. You have the statistical forecasting toolkit: exponential smoothing, ARIMA, the
methods for awkward seasonality, and now the means to say how confident any of them should be.

Part C turns to machine learning, which reframes forecasting as a supervised learning problem. The models
change completely; the baselines, the evaluation and the questions in this notebook do not.

**Solutions.** Worked answers to the 2 exercises above, with the reasoning behind them, are in
[B04_Probabilistic_forecasting_solutions.ipynb](../solutions/B04_Probabilistic_forecasting_solutions.ipynb). Try each one yourself first.
